In [ ]:
# ============================================================
# XGBOOST
# SIMPLE FROM-SCRATCH VERSION + SCIKIT-LEARN STYLE USAGE
# ============================================================

import numpy as np

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor


# ============================================================
# 1. CREATE DATASET
# ============================================================

X, y = make_regression(
    n_samples=300,
    n_features=2,
    noise=10,
    random_state=42
)


# ============================================================
# 2. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# 3. SIMPLE XGBOOST-STYLE REGRESSION
# ============================================================

class SimpleXGBoost:

    def __init__(
        self,
        n_estimators=20,
        learning_rate=0.1,
        reg_lambda=1.0
    ):

        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.reg_lambda = reg_lambda

        self.trees = []
        self.base_prediction = 0


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    def fit(self, X, y):

        # Start with mean prediction
        self.base_prediction = np.mean(y)

        predictions = np.full(
            len(y),
            self.base_prediction
        )

        for _ in range(self.n_estimators):

            # ------------------------------------------------
            # 1. Calculate gradient
            # ------------------------------------------------

            # For squared error:
            #
            # gradient = prediction - actual

            gradient = predictions - y


            # ------------------------------------------------
            # 2. Calculate Hessian
            # ------------------------------------------------

            # For squared error:
            #
            # Hessian = 1

            hessian = np.ones(len(y))


            # ------------------------------------------------
            # 3. Calculate leaf weight
            # ------------------------------------------------

            G = np.sum(gradient)

            H = np.sum(hessian)

            weight = -G / (H + self.reg_lambda)


            # ------------------------------------------------
            # 4. Store this tree/update
            # ------------------------------------------------

            self.trees.append(weight)


            # ------------------------------------------------
            # 5. Update prediction
            # ------------------------------------------------

            predictions += (
                self.learning_rate * weight
            )

        return self


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    def predict(self, X):

        predictions = np.full(
            len(X),
            self.base_prediction
        )

        for weight in self.trees:

            predictions += (
                self.learning_rate * weight
            )

        return predictions


# ============================================================
# 4. TRAIN SIMPLE MODEL
# ============================================================

# Gradient Boosting
#        ↓
# Sequential trees
#        ↓
# Correct errors

# XGBoost
#        ↓
# Sequential trees
#        ↓
# Gradient + Hessian
#        ↓
# Find best split using gain
#        ↓
# Regularize leaf weights
#        ↓
# Add tree × learning rate

scratch_model = SimpleXGBoost(
    n_estimators=20,
    learning_rate=0.1,
    reg_lambda=1.0
)

scratch_model.fit(
    X_train,
    y_train
)


# ============================================================
# 5. PREDICT
# ============================================================

scratch_pred = scratch_model.predict(
    X_test
)


# ============================================================
# 6. EVALUATE
# ============================================================

print(
    "Scratch MSE:",
    mean_squared_error(
        y_test,
        scratch_pred
    )
)

print(
    "Scratch R2:",
    r2_score(
        y_test,
        scratch_pred
    )
)


# ============================================================
# 7. REAL XGBOOST
# ============================================================

model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    objective="reg:squarederror",
    random_state=42
)


# ============================================================
# 8. TRAIN
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# 9. PREDICT
# ============================================================

pred = model.predict(
    X_test
)


# ============================================================
# 10. EVALUATE
# ============================================================

mse = mean_squared_error(
    y_test,
    pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    pred
)

print("XGBoost MSE:", mse)
print("XGBoost RMSE:", rmse)
print("XGBoost R2:", r2)


# ============================================================
# 11. EXPERIMENT WITH NUMBER OF TREES
# ============================================================

for n in [10, 50, 100, 200]:

    model = XGBRegressor(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "n_estimators:",
        n,
        "| R2:",
        score
    )


# ============================================================
# 12. EXPERIMENT WITH LEARNING RATE
# ============================================================

for lr in [0.01, 0.05, 0.1, 0.2]:

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "learning_rate:",
        lr,
        "| R2:",
        score
    )


# ============================================================
# 13. EXPERIMENT WITH MAX DEPTH
# ============================================================

for depth in [1, 2, 3, 5, 8]:

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "max_depth:",
        depth,
        "| R2:",
        score
    )


# ============================================================
# 14. IMPORTANT XGBOOST PARAMETERS
# ============================================================

# n_estimators
#
# Number of boosting trees.
#
# More trees → more learning steps.


# learning_rate
#
# Controls the contribution of each tree.
#
# Small learning_rate → usually need more trees.


# max_depth
#
# Maximum depth of each tree.
#
# Larger → more complex model
# Smaller → simpler model


# min_child_weight
#
# Minimum amount of Hessian weight required
# in a child node.
#
# Higher value → more conservative splitting.


# subsample
#
# Fraction of training samples used for each tree.
#
# Example:
#
# subsample=0.8
#
# → use 80% of training data per tree.


# colsample_bytree
#
# Fraction of features used by each tree.


# gamma
#
# Minimum loss reduction required
# to make a split.
#
# Higher gamma → fewer splits.


# reg_lambda
#
# L2 regularization on leaf weights.


# reg_alpha
#
# L1 regularization on leaf weights.


# objective
#
# Defines the learning objective.
#
# Example:
#
# reg:squarederror
# → regression with squared error


# ============================================================
# 15. XGBOOST CORE MATH
# ============================================================

# XGBoost uses:
#
#     Gradient + Hessian
#
# Gradient:
#
#     G = sum(gradients)
#
# Hessian:
#
#     H = sum(hessians)
#
#
# Simplified leaf weight:
#
#     w = -G / (H + lambda)
#
#
# This is one of the important differences
# from basic Gradient Boosting.


# ============================================================
# 16. SPLIT GAIN
# ============================================================

# For a potential split:
#
#        Parent
#       /      \
#    Left     Right
#
#
# Gain:
#
# 1/2 * [
#
#     G_left^2 / (H_left + lambda)
#
#     +
#
#     G_right^2 / (H_right + lambda)
#
#     -
#
#     G_parent^2 / (H_parent + lambda)
#
# ]
#
# - gamma
#
#
# The split with the highest positive gain
# is preferred.


# ============================================================
# 17. CORE WORKING
# ============================================================

# XGBoost:
#
# Current predictions
#        ↓
# Calculate gradients
#        ↓
# Calculate Hessians
#        ↓
# Try possible tree splits
#        ↓
# Calculate split gain
#        ↓
# Choose useful splits
#        ↓
# Calculate leaf weights
#        ↓
# Add tree using learning rate
#        ↓
# Repeat


# ============================================================
# 18. GRADIENT BOOSTING VS XGBOOST
# ============================================================

# Gradient Boosting:
#
#     Current prediction
#           ↓
#     Calculate residual / gradient
#           ↓
#     Train tree
#           ↓
#     Add tree
#
#
# XGBoost:
#
#     Current prediction
#           ↓
#     Gradient + Hessian
#           ↓
#     Optimize tree splits
#           ↓
#     Regularization
#           ↓
#     Calculate leaf weights
#           ↓
#     Add tree
#
#
# XGBoost adds:
#
#     1. Second-order optimization
#     2. Regularization
#     3. Better split optimization
#     4. Row/column subsampling
#     5. Engineering optimizations


# ============================================================
# 19. INTERVIEW SUMMARY
# ============================================================

# XGBoost is an optimized and regularized
# gradient boosting algorithm.
#
# Main ideas:
#
# Gradient
# Hessian
# Tree-based boosting
# Regularization
# Shrinkage
# Split gain
#
#
# Important parameters:
#
# n_estimators
# learning_rate
# max_depth
# min_child_weight
# subsample
# colsample_bytree
# gamma
# reg_alpha
# reg_lambda